# 01. 线性模型体系

线性模型是机器学习中最重要的一类基线模型。虽然形式简单，但它们几乎覆盖了：

- 回归问题中的基础建模思路
- 分类问题中的概率建模思路
- 正则化、可解释性、凸优化等核心概念

## 线性模型的形式化定义

            线性模型是一类以输入特征的线性组合为核心假设的统计学习模型。其基本形式可统一写为：

$$
f(x) = w^T x + b
$$

在线性回归中，$f(x)$ 直接作为实值预测；在 Logistic 回归中，$f(x)$ 作为 logit，再通过非线性链接函数映射到概率空间。线性模型的关键价值在于：模型假设清晰、目标函数通常是凸的、参数具有直接解释性。

## 输入、输出与参数化方式

            输入通常为定长特征向量 $x \in \mathbb{R}^d$，参数为权重向量 $w \in \mathbb{R}^d$ 与偏置标量 $b$。

- 回归任务输出 $\hat{y} \in \mathbb{R}$
- 二分类任务输出 $P(y=1|x)\in[0,1]$
- 多分类任务则输出各类别的概率分布

对于高维稀疏特征，线性模型仍然具有很强的适配能力，这也是它在工业 CTR、风控、文本稀疏特征场景长期被使用的重要原因。

## 结构分解与信息流

            线性模型的结构极简，但其统计含义非常完整：

1. 每个输入特征对应一个权重
2. 权重与特征相乘，表示该特征对预测的边际贡献
3. 所有贡献相加，再叠加偏置项
4. 根据任务类型，直接输出数值或再经由链接函数输出概率

在这种结构下，模型不会自动学习复杂层级表示，因此它更依赖输入特征的表达质量，而不是网络深度。

## 优化目标与训练机制

            线性回归通常最小化均方误差，Logistic 回归通常最小化负对数似然或交叉熵。加入 L1 / L2 正则化后，优化目标不仅关注拟合误差，还会对参数复杂度施加约束。

这使得线性模型天然适合作为“偏差-方差权衡”的教学起点：

- 无正则化：偏差较低，但更容易过拟合
- L2 正则化：参数整体收缩，模型更稳定
- L1 正则化：得到稀疏解，兼具拟合和特征选择作用

## 计算复杂度、统计性质与工程代价

            在线性模型中，单次预测复杂度通常为 $O(d)$；若采用闭式解，训练复杂度与矩阵求逆相关；若采用梯度法，则与样本数、特征数和迭代轮数线性相关。

从统计角度看，线性模型假设较强，因此在关系接近线性的场景中高效且稳健；在复杂非线性场景中则可能出现明显欠拟合。

## 与相邻模型的差异

            与决策树相比，线性模型的边界更平滑、可解释性更直接，但对复杂非线性关系表达不足。
与神经网络相比，线性模型的表示能力有限，但优化更稳定、训练更便宜、结果更容易解释。
因此在线上系统中，线性模型经常承担两个角色：一是强基线，二是高可解释性模型。

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis("off")
ax.set_xlim(0, 10)
ax.set_ylim(0, 4)

for idx, x in enumerate([1.0, 1.8, 2.6, 3.4]):
    circle = plt.Circle((x, 2), 0.22, color="#4C78A8")
    ax.add_patch(circle)
    ax.text(x, 1.45, f"x{idx+1}", ha="center", fontsize=12)

rect = plt.Rectangle((4.4, 1.25), 1.4, 1.5, facecolor="#F58518", edgecolor="black")
ax.add_patch(rect)
ax.text(5.1, 2.0, "加权求和\nw^T x + b", ha="center", va="center", fontsize=12)

rect2 = plt.Rectangle((6.6, 1.25), 1.2, 1.5, facecolor="#54A24B", edgecolor="black")
ax.add_patch(rect2)
ax.text(7.2, 2.0, "输出", ha="center", va="center", fontsize=12)
ax.text(7.2, 1.3, "ŷ 或 p(y|x)", ha="center", va="bottom", fontsize=11)

for x in [1.22, 2.02, 2.82, 3.62]:
    ax.annotate("", xy=(4.4, 2), xytext=(x, 2), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.annotate("", xy=(6.6, 2), xytext=(5.8, 2), arrowprops=dict(arrowstyle="->", lw=1.8))

ax.set_title("线性模型结构示意图")
plt.show()

## 学习目标

学完本 notebook 后，你应该能够：

1. 写出线性回归、Ridge、Lasso、Logistic 回归的目标函数。
2. 解释正则化为什么能降低过拟合。
3. 使用 `sklearn` 完成回归和分类任务。
4. 通过系数图、残差图、混淆矩阵理解模型行为。

## 先建立直觉

            可以把线性模型理解成“给每个特征一个分数，然后把这些分数加起来做决定”。

例如预测房价时：

- 面积可能加分
- 房龄可能减分
- 地段可能强烈加分

这些“加分”和“减分”的力度，就是模型学出来的参数。

线性模型最重要的价值不是“永远最强”，而是：

- 它是很多复杂模型的出发点
- 它的每个参数都容易解释
- 你很容易看懂模型为什么做出某个预测

## 架构是怎么工作的

            对回归问题，模型输出是一个实数；对分类问题，模型先算线性分数，再把分数转成概率。

从结构上看，线性模型几乎没有“深层结构”，它本质上只有一层：

1. 接收输入特征
2. 每个特征乘以自己的权重
3. 把结果加总，再加偏置
4. 输出预测值或类别概率

这也是为什么线性模型速度快、结构清晰、可解释性强。

但代价也很明显：

- 如果真实关系是弯曲的、复杂的、存在高阶交互，线性模型很难直接表达
- 它更依赖好的特征工程

## 训练时到底发生了什么

            训练线性模型，其实就是在问：

“怎样选择一组权重，让模型预测结果和真实答案尽量接近？”

对回归来说，通常最小化平方误差；对分类来说，通常最小化交叉熵。

正则化的意义也很重要：

- Ridge 会限制权重不要过大，让模型更稳
- Lasso 会主动把一部分权重压到 0，起到特征选择作用

所以训练线性模型并不只是拟合数据，更是在做“拟合能力”和“泛化能力”的平衡。

## 什么时候该用它

            优先考虑线性模型的场景：

- 你需要一个强基线
- 你希望看到每个特征对结果的影响方向和强度
- 数据规模不算大，但想先快速判断问题难度
- 你要做商业、金融、医学等需要解释性的任务

不适合的场景：

- 图像、语音、长文本等高维复杂结构数据
- 明显存在复杂非线性边界的问题

## 最常见的误区

            初学者最容易有三个误解：

1. **误以为“线性模型简单，所以没有学习价值”**
   实际上，很多优化、正则化、概率建模思想都要先从这里理解。

2. **误以为 Logistic 回归是回归模型**
   它名字里有“回归”，但最常见用途是分类，因为它输出的是类别概率。

3. **误以为线性模型一定只能拟合直线**
   只要你做了非线性特征变换，例如多项式特征，线性模型也能拟合复杂函数。

## 1. 线性回归的公式与原理

对于输入向量 $x \in \mathbb{R}^d$，线性回归假设：

$$
\hat{y} = w^T x + b
$$

其中：

- $w$ 是权重向量
- $b$ 是偏置项
- $\hat{y}$ 是模型预测值

在线性回归中，我们通常最小化均方误差：

$$
J(w, b) = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2
$$

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]



print("临时目录:", temp_root)

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

diabetes = load_diabetes(as_frame=True)
X_reg = diabetes.data
y_reg = diabetes.target

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

reg_models = {
    "LinearRegression": LinearRegression(),
    "Ridge(alpha=1.0)": Ridge(alpha=1.0),
    "Lasso(alpha=0.05)": Lasso(alpha=0.05),
}

reg_results = []
fitted_reg_models = {}

for name, model in reg_models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
    pipe.fit(X_train_reg, y_train_reg)
    preds = pipe.predict(X_test_reg)
    fitted_reg_models[name] = pipe
    reg_results.append(
        {
            "模型": name,
            "MSE": mean_squared_error(y_test_reg, preds),
            "MAE": mean_absolute_error(y_test_reg, preds),
            "R^2": r2_score(y_test_reg, preds),
        }
    )

reg_results_df = pd.DataFrame(reg_results).sort_values("MSE")
reg_results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=reg_results_df, x="模型", y="MSE", ax=axes[0], palette="Blues_d")
axes[0].set_title("不同线性回归模型的 MSE")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=reg_results_df, x="模型", y="R^2", ax=axes[1], palette="Greens_d")
axes[1].set_title("不同线性回归模型的 R^2")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

## 2. 正则化：Ridge 与 Lasso

### Ridge（L2 正则化）

$$
J(w, b) = \frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2 + \lambda \|w\|_2^2
$$

### Lasso（L1 正则化）

$$
J(w, b) = \frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2 + \lambda \|w\|_1
$$

Ridge 更倾向于整体收缩参数；Lasso 更容易得到稀疏解。

In [ ]:
coef_df = pd.DataFrame({"feature": X_reg.columns})
for name, pipe in fitted_reg_models.items():
    coef_df[name] = pipe.named_steps["model"].coef_

coef_long = coef_df.melt(id_vars="feature", var_name="模型", value_name="系数")

plt.figure(figsize=(14, 7))
sns.barplot(data=coef_long, x="feature", y="系数", hue="模型")
plt.xticks(rotation=45, ha="right")
plt.title("不同线性模型的特征系数对比")
plt.tight_layout()
plt.show()

In [ ]:
alphas = np.logspace(-3, 1, 20)
coef_path = []

for alpha in alphas:
    pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", Lasso(alpha=alpha, max_iter=10000)),
        ]
    )
    pipe.fit(X_train_reg, y_train_reg)
    coef_path.append(pipe.named_steps["model"].coef_)

coef_path = np.array(coef_path)

plt.figure(figsize=(12, 7))
for idx, feature_name in enumerate(X_reg.columns):
    plt.plot(alphas, coef_path[:, idx], label=feature_name)

plt.xscale("log")
plt.xlabel("alpha (log scale)")
plt.ylabel("coefficient value")
plt.title("Lasso 系数路径图")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 3. Logistic 回归

Logistic 回归先计算线性得分：

$$
z = w^T x + b
$$

再通过 Sigmoid 转成概率：

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

最终最小化的常见目标是二元交叉熵。

In [ ]:
from sklearn.datasets import load_breast_cancer, make_classification

X_syn, y_syn = make_classification(
    n_samples=400,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.8,
    random_state=42,
)

syn_model = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression())])
syn_model.fit(X_syn, y_syn)

xx, yy = np.meshgrid(
    np.linspace(X_syn[:, 0].min() - 1, X_syn[:, 0].max() + 1, 300),
    np.linspace(X_syn[:, 1].min() - 1, X_syn[:, 1].max() + 1, 300),
)
grid = np.c_[xx.ravel(), yy.ravel()]
prob = syn_model.predict_proba(grid)[:, 1].reshape(xx.shape)

plt.figure(figsize=(10, 8))
plt.contourf(xx, yy, prob, levels=20, cmap="RdBu", alpha=0.35)
plt.contour(xx, yy, prob, levels=[0.5], colors="black", linewidths=2)
plt.scatter(X_syn[:, 0], X_syn[:, 1], c=y_syn, cmap="Set1", edgecolor="k", s=40)
plt.title("Logistic 回归的决策边界与概率等高线")
plt.show()

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X_cls = cancer.data
y_cls = cancer.target

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

logreg = Pipeline(
    [("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=5000))]
)
logreg.fit(X_train_cls, y_train_cls)
y_pred_cls = logreg.predict(X_test_cls)

print(classification_report(y_test_cls, y_pred_cls, target_names=cancer.target_names))
print("Accuracy:", round(accuracy_score(y_test_cls, y_pred_cls), 4))
print("F1 score:", round(f1_score(y_test_cls, y_pred_cls), 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ConfusionMatrixDisplay.from_predictions(
    y_test_cls, y_pred_cls, display_labels=cancer.target_names, cmap="Blues", ax=axes[0]
)
axes[0].set_title("Breast Cancer: 混淆矩阵")

coef_series = pd.Series(
    logreg.named_steps["model"].coef_[0], index=X_cls.columns
).sort_values(key=np.abs, ascending=False).head(12)
sns.barplot(x=coef_series.values, y=coef_series.index, palette="vlag", ax=axes[1])
axes[1].set_title("Logistic 回归中最重要的 12 个特征")
axes[1].set_xlabel("系数")

plt.tight_layout()
plt.show()

## 4. 线性模型的用法总结

- 需要速度快、可解释、强基线时，优先考虑线性模型。
- 需要控制过拟合时，优先考虑正则化版本。
- 数据关系明显非线性时，线性模型往往不如树模型与神经网络。